In [1]:
from azureml.core import Workspace, Dataset, Datastore

# اتصال به Workspace
subscription_id = 'f8c5aac3-29fc-4387-858a-1f61722fb57a'
resource_group = 'forskerpl-n0ybkr-rg'
workspace_name = 'forskerpl-n0ybkr-mlw'

ws = Workspace(subscription_id=subscription_id,
               resource_group=resource_group,
               workspace_name=workspace_name)

# گرفتن datastore
datastore = Datastore.get(ws, "researcher_data")

# خواندن همه فایل‌های parquet در مسیر مشخص
dataset = Dataset.Tabular.from_parquet_files(
    path=[(datastore, 'Zahra/012026/Data/MEDS_MDPS/data/train/*.parquet')]    #     Zahra/Data-07-2025/MDP/MEDS_811/data/train
)

# تبدیل به pandas DataFrame
df = dataset.to_pandas_dataframe()
df.head(15)


/anaconda/envs/azureml_py310_sdkv2/lib/python3.10/site-packages/mlflow/__init__.py:41: UserWarning: Versions of mlflow (3.1.1) and mlflow-skinny (2.22.1) are different. This may lead to unexpected behavior. Please install the same version of both packages.
  mlflow.mismatch._check_version_mismatch()


Resolving access token for scope "https://storage.azure.com/.default" using identity of type "MANAGED".
Getting data access token with Assigned Identity (client_id=clientid) and endpoint type based on configuration
{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe'}
{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe', 'activityApp': 'TabularDataset'}


,subject_id,time,code,numeric_value
0,51,NaT,GENDER//Kvinde,NaN
1,51,1991-04-09 00:00:00,DOB,NaN
2,51,2018-05-29 00:00:00,D/DJ039,NaN
3,51,2020-02-19 10:57:00,P/UXUD10,NaN
4,51,2020-02-20 00:00:00,D/DK802,NaN
5,51,2020-02-24 08:25:00,P/AAF20,NaN
6,51,2020-02-24 08:25:00,P/ZZ0150,NaN
7,110,NaT,GENDER//Mand,NaN
8,110,1953-01-26 00:00:00,DOB,NaN
9,110,2016-05-31 08:08:00,P/UXCD60,NaN


In [2]:
len(df)

448503648

In [3]:
import pandas as pd

# فرض بر این که فایل CSV رو داری
# df = pd.read_csv("your_file.csv")  # یا مستقیم اگر DataFrame آماده‌ست، نیازی نیست

# دسته‌بندی هر subject_id بر اساس وجود کدهای مختلف
m_patients = df[df['code'].str.startswith('M/', na=False)]['subject_id'].unique()
p_patients = df[df['code'].str.startswith('P/', na=False)]['subject_id'].unique()
d_patients = df[df['code'].str.startswith('D/', na=False)]['subject_id'].unique()
s_patients = df[df['code'].str.startswith('S/', na=False)]['subject_id'].unique()

# کل بیماران منحصربه‌فرد
all_patients = df['subject_id'].unique()

# نمایش آمار
print(f"Whole Patients: {len(all_patients)}")
print(f"The patients has M-medication Codes: {len(m_patients)}")
print(f"The patients has D-diagnosis Codes: {len(d_patients)}")
print(f"The patients has P-Procedure Codes: {len(p_patients)}")
print(f"The patients has S-SKS Codes: {len(s_patients)}")


Whole Patients: 1774422
The patients has M-medication Codes: 1208927
The patients has D-diagnosis Codes: 1773917
The patients has P-Procedure Codes: 1701442
The patients has S-SKS Codes: 420586


In [4]:
subject_counts = df['subject_id'].value_counts()

In [5]:
subject_counts

921542     82107
698589     63432
2016077    60873
954669     58832
1964088    57150
           ...  
131767         2
1079388        2
1169509        2
1703144        2
484192         2
Name: subject_id, Length: 1774422, dtype: int64

In [6]:
s_Num = df[df['code'].str.startswith('S/', na=False)]

In [7]:
s_Num

,subject_id,time,code,numeric_value
29,110,2016-08-15 11:46:00,S/KNGB30,NaN
210,173,2018-12-26 14:59:00,S/KLCH00,NaN
598,175,2019-12-24 16:45:56,S/KNCJ45 KNCJ07,NaN
684,175,2020-12-05 09:36:18,S/KNCJ45,NaN
794,197,2020-10-23 09:01:03,S/KNAG74,NaN
...,...,...,...,...
448498448,2217292,2021-10-15 13:42:30,S/KCJE20,NaN
448500384,2217732,2021-07-23 11:02:07,S/KJKA21,NaN
448501145,2217877,2018-02-16 10:16:49,S/KNHK57,NaN
448501571,2217973,2017-06-20 14:01:14,S/KJAH01A,NaN


In [8]:
import pandas as pd

# پیدا کردن سطرهایی که فقط codeهای نوع /P دارن
only_p = df[df['code'].str.startswith('S/', na=False)]

# بیماران با فقط /P کد
subject_ids_only_p = only_p['subject_id'].unique()

# حالا بیماران با codeهای غیر از /P
not_p = df[~df['code'].str.startswith('S/', na=False)]
subject_ids_with_non_p = set(not_p['subject_id'].unique())

# حذف بیمارانی که فقط /P دارن
only_p_ids_to_exclude = [sid for sid in subject_ids_only_p if sid not in subject_ids_with_non_p]

print("Number of patients with only Srugery code: ", only_p_ids_to_exclude)


Number of patients with only Srugery code:  []


In [9]:
df_filtered = df[~df['code'].str.startswith('S/', na=False)]

In [10]:
subject_counts_MDS = df_filtered['subject_id'].value_counts()

In [11]:
subject_counts_MDS

921542     82103
698589     63432
2016077    60873
954669     58832
1964088    57148
           ...  
1079388        2
1428518        2
1169509        2
131767         2
1434427        2
Name: subject_id, Length: 1774422, dtype: int64

In [12]:
subject_counts_df = subject_counts.reset_index()
subject_counts_df.columns = ['subject_id', 'original_count']

subject_counts_MDS_df = subject_counts_MDS.reset_index()
subject_counts_MDS_df.columns = ['subject_id', 'new_count']


In [13]:
import pandas as pd
comparison_df = pd.merge(subject_counts_df, subject_counts_MDS_df, on='subject_id', how='outer')


In [14]:
comparison_df['difference'] =  comparison_df['original_count'] - comparison_df['new_count']


In [15]:
comparison_df = comparison_df.sort_values(by='difference', ascending=False)


In [16]:
comparison_df

,subject_id,original_count,new_count,difference
18677,312261,2863,2834,29
7922,1541863,4367,4339,28
1969,1281561,7969,7941,28
57353,2162677,1458,1430,28
7142,121479,4582,4556,26
...,...,...,...,...
83934,517748,1112,1112,0
850375,2145221,76,76,0
850374,768326,76,76,0
850373,1456317,76,76,0


In [17]:
unchanged_count = (comparison_df['difference'] == 0).sum()
print("unchanged_count", unchanged_count)


unchanged_count 1353836


In [18]:
comparison_df['abs_diff'] = comparison_df['difference'].abs()
most_changed = comparison_df.sort_values(by='abs_diff', ascending=False)


In [19]:
print(most_changed.head(10))


       subject_id  original_count  new_count  difference  abs_diff
18677      312261            2863       2834          29        29
1969      1281561            7969       7941          28        28
57353     2162677            1458       1430          28        28
7922      1541863            4367       4339          28        28
7142       121479            4582       4556          26        26
10057      736003            3910       3885          25        25
21825     1931799            2631       2607          24        24
19957      189251            2761       2739          22        22
23559     1755152            2522       2500          22        22
36323      342709            1960       1939          21        21


In [20]:
changed_df = comparison_df[comparison_df['difference'] != 0]
min_new_count = changed_df['new_count'].min()
lowest_new_count_patients = changed_df[changed_df['new_count'] == min_new_count]


In [21]:
lowest_new_count_patients = lowest_new_count_patients.rename(
    columns={
        'original_count': 'MDPS codes',
        'new_count': 'MDP codes'
    }
)


In [22]:
lowest_new_count_patients

,subject_id,MDPS codes,MDP codes,difference,abs_diff
1723311,451630,6,5,1,1
1723765,228541,6,5,1,1


.str.upper() شرط را case-insensitive می‌کند

بل از فیلتر، ستون را به pd.StringDtype() تبدیل می‌کند؛ این کار رفتار .str را پایدار و قابل پیش‌بینی می‌کند (<NA> به‌جای NaN)



In [23]:
# کل بیماران منحصربه‌فرد
all_patients = df_filtered['subject_id'].unique()
len(all_patients)
# نمایش آمار

1774422

In [24]:
len(df_filtered)

447890413

In [25]:
import pandas as pd
import numpy as np

# امن‌تر: اگر code نال یا غیررشته‌ای بود اذیت نکنه
df['code'] = df['code'].astype('string')
df_filtered = df[~df['code'].str.upper().str.startswith('S/', na=False)].copy()
print("kept rows MDP:", len(df_filtered), " / total:", len(df))


In [ ]:
import pandas as pd
import numpy as np

# اطمینان از نوع‌ها (برای خروجی تمیز و بدون خطا)
df_filtered = df_filtered.copy()
df_filtered['subject_id']    = pd.to_numeric(df_filtered['subject_id'], errors='coerce').astype('Int64')
df_filtered['numeric_value'] = pd.to_numeric(df_filtered['numeric_value'], errors='coerce').astype('float32')
df_filtered['time']          = pd.to_datetime(df_filtered['time'], errors='coerce', utc=False)

# ردیف‌های بدون subject_id را حذف کنیم (نمی‌توان شارد کرد)
df_filtered = df_filtered.dropna(subset=['subject_id']).copy()
df_filtered['subject_id'] = df_filtered['subject_id'].astype('int64')

# ۳۶ شارد: هر بیمار فقط در یک فایل (mod 36)
N_SHARDS = 45
df_filtered['__shard__'] = (df_filtered['subject_id'] % N_SHARDS).astype('int16')

print("rows to write:", len(df_filtered))


In [ ]:
import numpy as np
import os

N_SHARDS = 36   #45 for Whole # 36 when we have split
OUT_DIR = "./_TrainMDPS_withoutS_sharded"
os.makedirs(OUT_DIR, exist_ok=True)

cols_out = ['subject_id', 'time', 'code', 'numeric_value']

# فرض: subject_id قبلاً int64 شده و NaNها حذف شده‌اند (طبق سلول قبلی‌ات)
sid_mod = (df_filtered['subject_id'].to_numpy(dtype=np.int64, copy=False) % N_SHARDS)

written = 0
for k in range(N_SHARDS):
    mask = (sid_mod == k)                 # بدون ستون اضافی، فقط یک آرایهٔ NumPy
    part = df_filtered.loc[mask, cols_out].sort_values(['subject_id','time'])
    # اگر می‌خوای حتماً ۳۶ فایل 0..35 داشته باشی حتی اگه خالی باشن:
    # if part.empty:
    #     part = part.iloc[0:0]  # فایل صفر-سطر با همان ستون‌ها
    part.to_parquet(os.path.join(OUT_DIR, f"{k}.parquet"),
                    engine="pyarrow", compression="snappy", index=False)
    written += 1

print(f"Done. wrote {written} parquet files into {OUT_DIR}")


In [ ]:
import pyarrow.parquet as pq

OUT_DIR = "./_TrainMDPS_withoutS_sharded"

def parquet_num_rows(path):
    pf = pq.ParquetFile(path)
    md = pf.metadata
    return sum(md.row_group(i).num_rows for i in range(md.num_row_groups))

total = 0
for f in sorted(p for p in os.listdir(OUT_DIR) if p.endswith('.parquet')):
    n = parquet_num_rows(os.path.join(OUT_DIR, f))
    total += n
    print(f, "rows:", n)
print("TOTAL rows:", total)


In [ ]:
from azureml.data.datapath import DataPath
from azureml.data.dataset_factory import FileDatasetFactory
import os

OUT_DIR = "./_TrainMDPS_withoutS_sharded"
DST_PREFIX = "Zahra/012026/Data/MEDS_MDP/data/train"  #held_out"  # بدون اسلشِ اول
''

# اطمینان: پوشه خروجی وجود دارد و فایل parquet داخلش هست
print("Local files to upload:", len([f for f in os.listdir(OUT_DIR) if f.endswith(".parquet")]))

# مقصد روی Datastore
target = DataPath(datastore, DST_PREFIX)

# آپلود همه محتویات OUT_DIR به DST_PREFIX
_ = FileDatasetFactory.upload_directory(
    src_dir=OUT_DIR,
    target=target,
    overwrite=True,
    show_progress=True,
)
print(f"Uploaded to datastore path: {DST_PREFIX}")


In [ ]:
paths = Dataset.File.from_files(path=[(datastore, f"{DST_PREFIX}/*.parquet")]).to_path()
print("Found in datastore:", len(paths))
print(paths[:20])
